# Assignment 2: Bollinger Bands and RSI research
**Using:** LMT, RTX, FRO, STNG, INSW, AA, CENX, CF, MOS, IPI, GSL.

Default design: research on 2018–2023; reserve 2024–2025 for a later backtest. All date windows are start-inclusive, end-exclusive.

In [1]:
from AlgorithmImports import *
from QuantConnect.Research import QuantBook
from QuantConnect.Indicators import BollingerBands, RelativeStrengthIndex, MovingAverageType
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
plt.rcParams["figure.figsize"] = (13, 6)

ModuleNotFoundError: No module named 'AlgorithmImports'

## 1. Configuration and historical data
Adjustment incorporates splits and dividends into the historical price series; next-open returns below are research approximations, not actual fills or cash dividend accounting. The 400 calendar days before the research start let Wilder RSI settle beyond its minimum readiness period.

In [ ]:
TICKERS = ["LMT", "RTX", "FRO", "STNG", "INSW", "AA", "CENX", "CF", "MOS", "IPI", "GSL"]
RESEARCH_START = datetime(2018, 1, 1)
RESEARCH_END = datetime(2024, 1, 1)  # exclusive
RESERVED_TEST_START = datetime(2024, 1, 1)
RESERVED_TEST_END = datetime(2026, 1, 1)  # exclusive; NOT loaded here
WARMUP_CALENDAR_DAYS = 400
BB_PERIOD, BB_STD = 20, 2
RSI_FAST, RSI_SLOW = 14, 28
RSI_LOW, RSI_HIGH = 30, 70
HORIZONS = [5, 10, 20]  # trading sessions, not calendar days
MIN_EVENTS = 20  # descriptive flag, not a statistical significance threshold
CHART_BARS = 252  # most recent research observations; use None for all
FOCUS_TICKER = "STNG"  # change after reviewing the comparison tables

assert RESEARCH_START < RESEARCH_END <= RESERVED_TEST_START < RESERVED_TEST_END
assert HORIZONS and all(isinstance(h, int) and h > 0 for h in HORIZONS)
assert FOCUS_TICKER in TICKERS
qb = QuantBook()
symbols = {
    ticker: qb.add_equity(ticker, Resolution.DAILY,
        data_normalization_mode=DataNormalizationMode.ADJUSTED).symbol
    for ticker in TICKERS
}

In [ ]:
prices = {}
coverage_rows = []
for ticker, symbol in symbols.items():
    history = qb.history(symbol, RESEARCH_START - timedelta(days=WARMUP_CALENDAR_DAYS),
                         RESEARCH_END, Resolution.DAILY)
    if history.empty:
        raise ValueError(f"No history returned for {ticker}; check subscription and dates.")
    frame = history.xs(symbol, level=0).copy() if isinstance(history.index, pd.MultiIndex) else history.copy()
    frame.index = pd.DatetimeIndex(frame.index)
    if frame.index.tz is not None:
        frame.index = frame.index.tz_localize(None)
    frame = frame.sort_index()
    if frame.index.has_duplicates:
        raise ValueError(f"Duplicate history timestamps for {ticker}.")
    frame = frame.loc[frame.index < RESEARCH_END, ["open", "high", "low", "close", "volume"]]
    if frame[["open", "close"]].isna().any().any() or (frame[["open", "close"]] <= 0).any().any():
        raise ValueError(f"Missing or invalid prices for {ticker}; inspect history before proceeding.")
    prices[ticker] = frame
    sample = frame.loc[(frame.index >= RESEARCH_START) & (frame.index < RESEARCH_END)]
    if sample.empty:
        raise ValueError(f"No research-period observations for {ticker}.")
    coverage_rows.append({"ticker": ticker, "first_research_bar": sample.index.min(),
                          "last_research_bar": sample.index.max(), "research_bars": len(sample),
                          "warmup_bars": int((frame.index < RESEARCH_START).sum())})
coverage = pd.DataFrame(coverage_rows).set_index("ticker")
display(coverage)
print("Timestamps are QC daily bar timestamps; all horizons use observed trading bars.")
print("No missing observations are forward-filled. Inspect differences in coverage before comparing stocks.")

## 2. Calculate the indicators
Bollinger Bands use a 20-session simple moving average ±2 standard deviations. RSI uses Wilder smoothing at 14 and 28 sessions. 

Indicators are updated chronologically with each close; values are retained only when all indicators are ready.

In [ ]:
def calculate_indicators(frame):
    bb = BollingerBands(BB_PERIOD, BB_STD, MovingAverageType.SIMPLE)
    fast = RelativeStrengthIndex(RSI_FAST, MovingAverageType.WILDERS)
    slow = RelativeStrengthIndex(RSI_SLOW, MovingAverageType.WILDERS)
    rows = []
    for stamp, close in frame["close"].items():
        time = stamp.to_pydatetime()
        bb.update(time, float(close))
        fast.update(time, float(close))
        slow.update(time, float(close))
        if bb.is_ready and fast.is_ready and slow.is_ready:
            rows.append({"time": stamp,
                         "bb_lower": float(bb.lower_band.current.value),
                         "bb_middle": float(bb.middle_band.current.value),
                         "bb_upper": float(bb.upper_band.current.value),
                         "rsi14": float(fast.current.value),
                         "rsi28": float(slow.current.value)})
    if not rows:
        raise ValueError("Not enough history to initialize indicators.")
    indicators = pd.DataFrame(rows).set_index("time")
    result = frame.join(indicators)
    width = result["bb_upper"] - result["bb_lower"]
    result["bb_percent_b"] = (result["close"] - result["bb_lower"]) / width.replace(0, np.nan)
    result["bb_width_pct"] = 100 * width / result["bb_middle"]
    result = result.loc[(result.index >= RESEARCH_START) & (result.index < RESEARCH_END)].copy()
    if result[["bb_lower", "bb_middle", "bb_upper", "rsi14", "rsi28"]].isna().any().any():
        raise ValueError("Increase warm-up history; indicators are not ready at the research start.")
    return result

research = {ticker: calculate_indicators(frame) for ticker, frame in prices.items()}
print(f"Calculated indicators for {len(research)} stocks.")
display(research[FOCUS_TICKER].tail().round(3))

In [ ]:
def plot_stock(ticker, bars=CHART_BARS):
    df = research[ticker] if bars is None else research[ticker].tail(bars)
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                             gridspec_kw={"height_ratios": [2, 1]})
    ax = axes[0]
    ax.plot(df.index, df["close"], label="Adjusted close", color="black", linewidth=1)
    ax.plot(df.index, df["bb_middle"], label=f"SMA {BB_PERIOD}", color="royalblue")
    ax.plot(df.index, df["bb_upper"], label="Upper band", color="gray", linewidth=.8)
    ax.plot(df.index, df["bb_lower"], label="Lower band", color="gray", linewidth=.8)
    ax.fill_between(df.index, df["bb_lower"].to_numpy(), df["bb_upper"].to_numpy(), alpha=.12)
    combined = (df["close"] < df["bb_lower"]) & (df["rsi14"] < RSI_LOW)
    ax.scatter(df.index[combined], df.loc[combined, "close"], color="crimson", s=22,
               label=f"Below lower band AND RSI{RSI_FAST} < {RSI_LOW}", zorder=4)
    combined_above = (df["close"] > df["bb_upper"]) & (df["rsi14"] > RSI_HIGH)
    ax.scatter(df.index[combined_above], df.loc[combined_above, "close"], color="dodgerblue", s=22,
               label=f"Above upper band AND RSI{RSI_FAST} > {RSI_HIGH}", zorder=4)
    ax.set_title(f"{ticker}: Bollinger Bands and RSI — research period")
    ax.set_ylabel("Adjusted price ($)")
    ax.legend(loc="best", fontsize=9)
    axes[1].plot(df.index, df["rsi14"], label=f"RSI {RSI_FAST}")
    axes[1].plot(df.index, df["rsi28"], label=f"RSI {RSI_SLOW}", alpha=.8)
    for threshold in [RSI_LOW, 50, RSI_HIGH]:
        axes[1].axhline(threshold, color="gray", linestyle="--", linewidth=.7)
    axes[1].set_ylim(0, 100)
    axes[1].set_ylabel("RSI")
    axes[1].legend(loc="best")
    for axis in axes:
        axis.grid(alpha=.2)
    fig.tight_layout()
    plt.show()

for ticker in TICKERS:
    plot_stock(ticker)

## 3. Define signals and compare subsequent returns
An event is the first bar when a condition becomes true, which avoids counting every day in a prolonged oversold episode as a fresh signal.

For each event at bar *t*, hypothetical entry is the **next observed bar's open**, and exit is the close of bar *t + h*, where *h* is 5, 10, or 20 sessions. This avoids assuming execution at the close used to calculate the signal. Forward returns that extend beyond the research boundary are excluded. The baseline uses the same return definition on all eligible research bars for that stock.

Returns are gross, unlevered, and descriptive. Events can still have overlapping holding windows, so counts are not counts of independent observations or executable portfolio trades. No costs, sizing, stop losses, or cash management are modeled here.

In [ ]:
def signal_conditions(df):
    return pd.DataFrame({
        "BB lower": df["close"] < df["bb_lower"],
        "BB upper": df["close"] > df["bb_upper"],
        "RSI14 low": df["rsi14"] < RSI_LOW,
        "RSI14 high": df["rsi14"] > RSI_HIGH,
        "RSI28 low": df["rsi28"] < RSI_LOW,
        "BB lower + RSI14 low": (df["close"] < df["bb_lower"]) & (df["rsi14"] < RSI_LOW),
        "BB upper + RSI14 high": (df["close"] > df["bb_upper"]) & (df["rsi14"] > RSI_HIGH),
        "RSI14 low + RSI28 > 50": (df["rsi14"] < RSI_LOW) & (df["rsi28"] > 50),
    }, index=df.index)

def event_starts(conditions):
    return conditions & ~conditions.shift(1, fill_value=False)

def forward_returns(df, horizon):
    return df["close"].shift(-horizon) / df["open"].shift(-1) - 1

conditions = {ticker: signal_conditions(df) for ticker, df in research.items()}
events = {ticker: event_starts(flags) for ticker, flags in conditions.items()}
# First visible bar has no preceding research bar; do not call it a new event.
for flags in events.values():
    flags.iloc[0] = False

frequency = pd.DataFrame([
    {"ticker": ticker, "signal": signal, "condition_days": int(flags[signal].sum()),
     "condition_pct": 100 * flags[signal].mean(), "event_starts": int(events[ticker][signal].sum())}
    for ticker, flags in conditions.items() for signal in flags.columns
])
display(frequency.round(2))

In [ ]:
def summarize_events(research, events):
    rows = []
    for ticker, df in research.items():
        for horizon in HORIZONS:
            returns = forward_returns(df, horizon)
            baseline = returns.dropna()
            for signal in events[ticker].columns:
                sample = returns.loc[events[ticker][signal]].dropna()
                rows.append({
                    "ticker": ticker, "signal": signal, "horizon": horizon,
                    "n_events": len(sample), "baseline_n": len(baseline),
                    "mean_pct": 100 * sample.mean(), "median_pct": 100 * sample.median(),
                    "positive_pct": 100 * (sample > 0).mean() if len(sample) else np.nan,
                    "p10_pct": 100 * sample.quantile(.1), "p90_pct": 100 * sample.quantile(.9),
                    "baseline_mean_pct": 100 * baseline.mean(),
                    "lift_pp": 100 * (sample.mean() - baseline.mean()),
                    "sample_flag": "Enough to inspect" if len(sample) >= MIN_EVENTS else "Sparse",
                })
    return pd.DataFrame(rows)

summary = summarize_events(research, events)
# Every stock is visible even if the combined condition never occurs.
comparison = summary.loc[(summary["signal"] == "BB lower + RSI14 low") &
                         (summary["horizon"] == 10)].set_index("ticker").reindex(TICKERS)
display(comparison.drop(columns=["signal", "horizon"]).round(2))
print("lift_pp = event mean return minus this stock's unconditional mean, in percentage points.")
print("A positive lift is a candidate observation, not evidence of statistically reliable alpha.")

In [ ]:
# Explore all signals for the selected stock.
focus_summary = summary.loc[summary["ticker"] == FOCUS_TICKER]
display(focus_summary.sort_values(["signal", "horizon"]).round(2))

### Signal comparison heatmaps
Each cell shows **lift in percentage points**: the average forward return after an event minus the same stock's unconditional average forward return. Blue means higher than baseline; red means lower. Each annotation also shows the number of eligible events (`n`). Gray cells have fewer than `MIN_EVENTS` events

There is one chart for each configured horizon (5, 10, and 20 trading sessions by default). All charts share a color scale so their colors are comparable. These are subsequent **long** returns, including for upper-band/high-RSI signals;

In [ ]:
def annotated_lift_heatmap(values, counts, title, color_limit, min_events):
    values, counts = values.align(counts, join="outer")
    eligible = (counts >= min_events) & values.notna()
    colored = values.where(eligible).to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(max(10, 1.8 * len(values.columns)),
                                    max(5, .65 * len(values.index))))
    cmap = plt.get_cmap("RdBu").copy()
    cmap.set_bad("#eeeeee")
    im = ax.imshow(np.ma.masked_invalid(colored), cmap=cmap,
                   vmin=-color_limit, vmax=color_limit, aspect="auto")
    ax.set_xticks(range(len(values.columns)))
    ax.set_xticklabels(values.columns, rotation=35, ha="right")
    ax.set_yticks(range(len(values.index)))
    ax.set_yticklabels(values.index)
    for row in range(len(values.index)):
        for col in range(len(values.columns)):
            count = counts.iloc[row, col]
            count = 0 if pd.isna(count) else int(count)
            value = values.iloc[row, col]
            if eligible.iloc[row, col]:
                label = f"{value:+.2f} pp\nn={count}"
                ink = "white" if abs(value) > .6 * color_limit else "black"
            else:
                label = f"Sparse\nn={count}" if count else "No events\nn=0"
                ink = "#555555"
            ax.text(col, row, label, ha="center", va="center", fontsize=8, color=ink)
    ax.set_title(title + f"\nGray = fewer than {min_events} events; gross research returns")
    fig.colorbar(im, ax=ax, label="Mean return advantage vs baseline (percentage points)")
    fig.tight_layout()
    plt.show()

signal_order = list(events[TICKERS[0]].columns)
eligible_lifts = summary.loc[summary["n_events"] >= MIN_EVENTS, "lift_pp"].dropna()
signal_color_limit = max(float(eligible_lifts.abs().max()), .1) if len(eligible_lifts) else 1.0
for horizon in HORIZONS:
    subset = summary.loc[summary["horizon"] == horizon]
    values = subset.pivot(index="ticker", columns="signal", values="lift_pp").reindex(
        index=TICKERS, columns=signal_order)
    counts = subset.pivot(index="ticker", columns="signal", values="n_events").reindex(
        index=TICKERS, columns=signal_order)
    annotated_lift_heatmap(values, counts,
                          f"Signal comparison: {horizon}-session return lift",
                          signal_color_limit, MIN_EVENTS)

### Does the pattern recur across years?
Compare annual results rather than relying only on the pooled average. Years with no events remain visible with zero counts and missing event returns. This is a descriptive stability check, not an independent validation set. Later test data remain reserved.

In [ ]:
annual_rows = []
SIGNAL_TO_INSPECT = "BB lower + RSI14 low"
HORIZON_TO_INSPECT = 10
for ticker, df in research.items():
    returns = forward_returns(df, HORIZON_TO_INSPECT)
    flags = events[ticker][SIGNAL_TO_INSPECT]
    for year in sorted(df.index.year.unique()):
        in_year = df.index.year == year
        sample = returns.loc[in_year & flags].dropna()
        baseline = returns.loc[in_year].dropna()
        annual_rows.append({"ticker": ticker, "year": int(year), "n_events": len(sample),
                            "mean_pct": 100 * sample.mean(),
                            "baseline_mean_pct": 100 * baseline.mean(),
                            "lift_pp": 100 * (sample.mean() - baseline.mean())})
annual = pd.DataFrame(annual_rows)
display(annual.pivot(index="ticker", columns="year", values="n_events").reindex(TICKERS))
display(annual.pivot(index="ticker", columns="year", values="lift_pp").reindex(TICKERS).round(2))